In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
from ultralytics import YOLO
from pathlib import Path
import time
from collections import defaultdict

class BirdTracker:
    def __init__(self, model_path, conf_threshold=0.3):
        self.model = YOLO(model_path)
        self.conf_threshold = conf_threshold
        print("Trained model loaded successfully. Let's start tracking! 🚀")
        
    def process_video_with_tracking(self, video_path, output_dir):
        """Process video with object tracking and save results"""
        # Create output directory
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        
        video_path = Path(video_path)
        video_name = video_path.name
        
        # Open video
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise ValueError(f"Error opening video file: {video_path}")
            
        # Get video properties
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Define output video writer
        output_video_path = output_path / video_name
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(str(output_video_path), fourcc, fps, (frame_width, frame_height))
        
        # Initialize tracking
        all_detections = []
        frame_number = 0
        track_history = defaultdict(lambda: [])
        
        print(f"Processing video: {video_name} ({total_frames} frames)")
        start_time = time.time()
        
        # Process video frame by frame
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
                
            # Perform tracking
            results = self.model.track(
                frame, 
                persist=True, 
                conf=self.conf_threshold,
                verbose=False  # Reduces console output for better performance
            )
            
            # Get the first result (since we're processing one frame at a time)
            if results and results[0].boxes is not None:
                result = results[0]
                frame_with_boxes = result.plot()  # Draw boxes and IDs
                
                # Extract tracking information
                if result.boxes.id is not None:
                    track_ids = result.boxes.id.int().cpu().tolist()
                    xyxy_coords = result.boxes.xyxy.cpu().tolist()
                    confidences = result.boxes.conf.cpu().tolist()
                    class_ids = result.boxes.cls.cpu().tolist()
                    
                    for i, track_id in enumerate(track_ids):
                        # Store detection data
                        detection_info = {
                            "frame": frame_number,
                            "track_id": track_id,
                            "x1": xyxy_coords[i][0],
                            "y1": xyxy_coords[i][1],
                            "x2": xyxy_coords[i][2],
                            "y2": xyxy_coords[i][3],
                            "confidence": confidences[i],
                            "class": class_ids[i],
                            "center_x": (xyxy_coords[i][0] + xyxy_coords[i][2]) / 2,
                            "center_y": (xyxy_coords[i][1] + xyxy_coords[i][3]) / 2
                        }
                        all_detections.append(detection_info)
                        
                        # Update track history for visualization
                        track = track_history[track_id]
                        center = (detection_info["center_x"], detection_info["center_y"])
                        track.append(center)
                        
                        # Draw trail paths (last 30 points)
                        if len(track) > 30:
                            track.pop(0)
                        
                        # Draw the tracking lines
                        points = np.array(track, dtype=np.int32).reshape((-1, 1, 2))
                        cv2.polylines(frame_with_boxes, [points], isClosed=False, color=(0, 255, 255), thickness=2)
            else:
                frame_with_boxes = frame
                
            # Write frame to output video
            out.write(frame_with_boxes)
            
            # Progress reporting
            if frame_number % 100 == 0:
                elapsed = time.time() - start_time
                fps_processed = frame_number / elapsed if elapsed > 0 else 0
                remaining = (total_frames - frame_number) / fps_processed if fps_processed > 0 else 0
                print(f"Processed frame {frame_number}/{total_frames} "
                      f"({frame_number/total_frames*100:.1f}%), "
                      f"ETA: {remaining:.1f}s")
            
            frame_number += 1
        
        # Release resources
        cap.release()
        out.release()
        print(f"Output video with tracking saved to: {output_video_path}")
        
        # Save detections to CSV
        csv_path = output_path / f"{video_path.stem}_detections_with_ids.csv"
        df_detections = pd.DataFrame(all_detections)
        df_detections.to_csv(csv_path, index=False)
        print(f"All bird detections with track IDs saved to: {csv_path}")
        
        # Calculate and print some statistics
        unique_birds = df_detections['track_id'].nunique()
        print(f"Found {unique_birds} unique birds in the video")
        
        return csv_path

# Main execution
if __name__ == "__main__":
    # Configuration
    BEST_MODEL_PATH = "D:/Project_P1/src/ml/runs/detect/train2/weights/best.pt"
    video_file_path = "D:/Project_P1/data/valid/test1.mp4"
    video_output_dir = "D:/Project_P1/results/video_analysis"
    
    # Initialize tracker
    tracker = BirdTracker(BEST_MODEL_PATH, conf_threshold=0.3)
    
    # Process video
    print("\n--- Running tracking on a video ---")
    csv_path = tracker.process_video_with_tracking(video_file_path, video_output_dir)
    print("\nTracking complete. The CSV file with unique bird IDs is ready for the prediction script.")

In [4]:
import os
import cv2
import pandas as pd
import numpy as np
import tensorflow as tf
from collections import deque, defaultdict
import random
import time

# --- 1. Configuration ---

# CORE PATHS
DETECTIONS_CSV_PATH = "D:/Project_P1/results/video_analysis/test1_detections_with_ids.csv"
VIDEO_PATH = "D:/Project_P1/data/valid/test1.mp4"
TFLITE_MODEL_PATH = "D:/Project_P1/results/video_analysis/model.tflite"
OUTPUT_VIDEO_PATH = "D:/Project_P1/results/video_analysis/video_with_multi_bird_predictions.mp4"

# MODEL & SEQUENCE PARAMETERS
SEQUENCE_LENGTH = 15
PREDICTION_HORIZON = 5  # Number of future points to predict

# PERFORMANCE THRESHOLDS (for real-time analysis)
MAX_PROCESSING_TIME_MS = 33  # ~30 FPS target (33ms per frame)
PREDICTION_TIME_WARNING_THRESHOLD_MS = 10  # Warning if prediction takes longer than this

# VISUALIZATION PARAMETERS
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.5
PREDICTION_THICKNESS = 3
DOT_RADIUS = 4

def generate_colors(num_colors):
    """Generates a list of distinct BGR colors for tracking."""
    colors = []
    for i in range(num_colors):
        r = random.randint(50, 255)
        g = random.randint(50, 255)
        b = random.randint(50, 255)
        colors.append((b, g, r))
    return colors

# --- 2. Flight Pattern Recognition ---

def analyze_multi_bird_patterns(df):
    """
    Analysis of kinematics for each track_id to classify its flight pattern.
    Returns detailed metrics for each bird.
    """
    patterns = {}
    metrics = {}
    
    for track_id, group in df.groupby('track_id'):
        if len(group) < 5:  # Need enough data points for a reliable analysis
            continue
            
        seq_data = group.sort_values('frame').copy()
        
        # Calculate derivatives (velocity and acceleration)
        x, y, frames = seq_data['center_x'].values, seq_data['center_y'].values, seq_data['frame'].values
        
        # Calculate time-based derivatives (assuming constant frame rate)
        dx, dy = np.gradient(x), np.gradient(y)
        velocity = np.sqrt(dx**2 + dy**2)
        
        ddx, ddy = np.gradient(dx), np.gradient(dy)
        acceleration = np.sqrt(ddx**2 + ddy**2)
        
        # Calculate jerk (rate of change of acceleration)
        dddx, dddy = np.gradient(ddx), np.gradient(ddy)
        jerk = np.sqrt(dddx**2 + dddy**2)
        
        # Calculate turning angles
        directions = np.arctan2(dy, dx)
        turning_angles = np.abs(np.diff(directions))
        turning_angles = np.append(turning_angles, 0)  # Maintain same length
        
        # Statistical analysis
        avg_vel, std_vel = np.mean(velocity), np.std(velocity)
        avg_acc, std_acc = np.mean(acceleration), np.std(acceleration)
        avg_jerk = np.mean(jerk)
        avg_turn = np.mean(turning_angles)
        
        # Classify the pattern based on statistical thresholds
        pattern_type = "Unknown"
        if std_vel < 5 and avg_acc < 2 and avg_turn < 0.2:
            pattern_type = "Steady flight"
        elif avg_acc > 5 or avg_turn > 0.5:
            pattern_type = "Maneuvering"
        elif std_vel > 10 or avg_jerk > 3:
            pattern_type = "Erratic flight"
        
        patterns[track_id] = pattern_type
        metrics[track_id] = {
            'avg_velocity': avg_vel,
            'std_velocity': std_vel,
            'avg_acceleration': avg_acc,
            'std_acceleration': std_acc,
            'avg_jerk': avg_jerk,
            'avg_turning_angle': avg_turn,
            'frames_tracked': len(group)
        }
        
    return patterns, metrics

# --- 3. Prediction Visualization Functions ---

def draw_prediction_path(frame, current_pos, predicted_points, color):
    """Draw only the future predicted path with dots and connecting lines."""
    if len(predicted_points) == 0:
        return
    
    # Convert all points to integers for drawing
    all_points = [tuple(map(int, current_pos))]
    for point in predicted_points:
        all_points.append(tuple(map(int, point)))
    
    # Draw connecting lines
    for i in range(len(all_points) - 1):
        cv2.line(frame, all_points[i], all_points[i+1], color, PREDICTION_THICKNESS)
    
    # Draw dots at each point
    for point in all_points:
        cv2.circle(frame, point, DOT_RADIUS, color, -1)

print("--- Phase 1: Pre-Analysis of Bird Detections ---")
try:
    df_detections = pd.read_csv(DETECTIONS_CSV_PATH)
    df_detections['center_x'] = (df_detections['x1'] + df_detections['x2']) / 2
    df_detections['center_y'] = (df_detections['y1'] + df_detections['y2']) / 2
except FileNotFoundError:
    print(f"ERROR: Detections file not found at {DETECTIONS_CSV_PATH}"); exit()

# Analyze patterns for all birds
track_patterns, track_metrics = analyze_multi_bird_patterns(df_detections)

print(f"Identified {df_detections['track_id'].nunique()} unique bird tracks.")
print("\n=== BIRD MOVEMENT ANALYSIS ===")
for tid, metrics in track_metrics.items():
    pattern = track_patterns.get(tid, "N/A")
    print(f"Bird {tid}: {pattern}")
    print(f"  Avg Velocity: {metrics['avg_velocity']:.2f} px/frame")
    print(f"  Velocity STD: {metrics['std_velocity']:.2f} px/frame")
    print(f"  Avg Acceleration: {metrics['avg_acceleration']:.2f} px/frame²")
    print(f"  Avg Jerk: {metrics['avg_jerk']:.2f} px/frame³")
    print(f"  Avg Turning Angle: {metrics['avg_turning_angle']:.2f} rad")
    print(f"  Frames Tracked: {metrics['frames_tracked']}")
    print()

# --- 4. Setup Video Processing with TFLite ---

print("\n--- Phase 2: Setting up Video Processing ---")
# Group detections by frame for quick multi-bird lookup
detections_by_frame = {
    frame: {row['track_id']: {
        'position': np.array([row['center_x'], row['center_y']]),
        'bbox': np.array([row['x1'], row['y1'], row['x2'], row['y2']])
    } for _, row in group.iterrows()}
    for frame, group in df_detections.groupby('frame')
}
   
# Load the TFLite model
try:
    interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    print("TFLite model loaded successfully.")
except Exception as e:
    print(f"ERROR: Failed to load TFLite model. Details: {e}"); exit()

# Setup video I/O
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened(): print(f"ERROR: Could not open video file at {VIDEO_PATH}"); exit()
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))

# --- 5. Process Video with Multi-Bird Prediction ---

print("\n--- Phase 3: Processing Video ---")
frame_number = 0

# Performance tracking
frame_times = []
prediction_times = []
track_response_times = {}  # {track_id: [first_detection_frame, first_prediction_frame]}

# Dictionaries to hold data for each track ID
track_histories = {}    # {track_id: deque([...])}
track_predictions = {}  # {track_id: [predicted_points]}
track_colors = {}       # {track_id: (B, G, R)}
track_first_seen = {}   # {track_id: first_frame_detected}
color_palette = generate_colors(50)  # Generate 50 random colors

# Process every frame (no dynamic frame skipping)
while cap.isOpened():
    frame_start_time = time.perf_counter()
    ret, frame = cap.read()
    if not ret: break

    current_frame_detections = detections_by_frame.get(frame_number, {})

    # Loop through each detected bird in the current frame
    for track_id, detection_data in current_frame_detections.items():
        position = detection_data['position']
        
        # Record when this bird was first detected
        if track_id not in track_first_seen:
            track_first_seen[track_id] = frame_number
        
        # Assign a unique color to each new bird track
        if track_id not in track_colors:
            track_colors[track_id] = color_palette[len(track_colors) % len(color_palette)]

        # Get or create the path history for this specific bird
        if track_id not in track_histories:
            track_histories[track_id] = deque(maxlen=SEQUENCE_LENGTH)
        
        # Add the current position to this bird's history
        track_histories[track_id].append(position)
        
        # Make prediction when we have enough history
        if len(track_histories[track_id]) == SEQUENCE_LENGTH:
            history = np.array(track_histories[track_id], dtype=np.float32)
            
            # Normalize the history
            normalized_history = history / [frame_width, frame_height]
            normalized_history = normalized_history.reshape(1, SEQUENCE_LENGTH, 2)
            
            # Run inference and time it
            prediction_start_time = time.perf_counter()
            interpreter.set_tensor(input_details[0]['index'], normalized_history.astype(np.float32))
            interpreter.invoke()
            predicted_normalized = interpreter.get_tensor(output_details[0]['index'])
            prediction_end_time = time.perf_counter()
            
            # Record prediction time
            prediction_time_ms = (prediction_end_time - prediction_start_time) * 1000
            prediction_times.append(prediction_time_ms)
            
            # Record response time (first prediction after detection)
            if track_id not in track_response_times:
                track_response_times[track_id] = {
                    'detection_frame': track_first_seen[track_id],
                    'prediction_frame': frame_number,
                    'response_time_frames': frame_number - track_first_seen[track_id]
                }
            
            # Convert back to pixel coordinates and store the full predicted path
            predicted_path = predicted_normalized[0] * [frame_width, frame_height]
            track_predictions[track_id] = predicted_path

    # --- Visualization: Draw predictions for all active tracks ---
    active_track_ids = set(current_frame_detections.keys())
    
    # Draw predictions for all active birds
    for track_id in active_track_ids:
        if track_id in track_histories and len(track_histories[track_id]) > 0:
            color = track_colors.get(track_id, (255, 0, 255))
            
            # Draw prediction if available
            if track_id in track_predictions:
                current_pos = track_histories[track_id][-1]
                draw_prediction_path(frame, current_pos, track_predictions[track_id], color)
                
                # Add label with bird ID and pattern
                pattern = track_patterns.get(track_id, "Unknown")
                label = f"Bird {track_id} ({pattern})"
                label_pos = (int(current_pos[0]) + 10, int(current_pos[1]) - 10)
                cv2.putText(frame, label, label_pos, FONT, FONT_SCALE, color, 2)

    # Display performance metrics on frame
    if prediction_times:
        recent_pred_times = prediction_times[-10:] if len(prediction_times) > 10 else prediction_times
        avg_pred_time = np.mean(recent_pred_times)
        cv2.putText(frame, f"Avg Pred Time: {avg_pred_time:.2f}ms", (10, 30), FONT, FONT_SCALE, (0, 255, 0), 2)
    
    cv2.putText(frame, f"Frame: {frame_number}/{total_frames}", (10, 60), FONT, FONT_SCALE, (0, 255, 0), 2)
    cv2.putText(frame, f"Active Birds: {len(active_track_ids)}", (10, 90), FONT, FONT_SCALE, (0, 255, 0), 2)

    out.write(frame)
    
    # Track frame processing time
    frame_end_time = time.perf_counter()
    frame_time_ms = (frame_end_time - frame_start_time) * 1000
    frame_times.append(frame_time_ms)
    
    if frame_number > 0 and frame_number % 100 == 0:
        avg_frame_time = np.mean(frame_times[-100:]) if len(frame_times) > 100 else np.mean(frame_times)
        print(f"   Processed frame {frame_number}/{total_frames} - Avg Frame Time: {avg_frame_time:.2f}ms")
    frame_number += 1

# --- 6. Finalize and Report ---
print("\n--- Phase 4: Finalizing and Reporting ---")
cap.release()
out.release()
cv2.destroyAllWindows()
print(f"\n✅ Success! Video with multi-bird dynamic predictions saved to:\n{OUTPUT_VIDEO_PATH}")

# Performance analysis
if frame_times:
    print("\n--- SYSTEM PERFORMANCE ANALYSIS ---")
    print(f"Total Frames Processed: {len(frame_times)}")
    print(f"Average Frame Processing Time: {np.mean(frame_times):.2f} ms")
    print(f"Maximum Frame Processing Time: {np.max(frame_times):.2f} ms")
    print(f"Minimum Frame Processing Time: {np.min(frame_times):.2f} ms")
    
    # Real-time capability assessment
    target_frame_time = 1000 / fps if fps > 0 else 33  # Default to 30 FPS if unknown
    real_time_factor = target_frame_time / np.mean(frame_times) if np.mean(frame_times) > 0 else 0
    print(f"Real-time Performance Factor: {real_time_factor:.2f} (>{1.0} means real-time capable)")
    
    if real_time_factor >= 1.0:
        print("🟢 REAL-TIME CAPABLE: System can process video in real-time")
    else:
        print("🔴 NOT REAL-TIME: System cannot keep up with real-time processing")
        print(f"   Needed: {target_frame_time:.2f}ms per frame, Achieved: {np.mean(frame_times):.2f}ms per frame")

if prediction_times:
    print("\n--- TFLite Model Performance ---")
    print(f"Total Predictions Made: {len(prediction_times)}")
    print(f"Average Prediction Time: {np.mean(prediction_times):.2f} ms")
    print(f"Maximum Prediction Time: {np.max(prediction_times):.2f} ms")
    print(f"Minimum Prediction Time: {np.min(prediction_times):.2f} ms")
    
    if np.mean(prediction_times) > PREDICTION_TIME_WARNING_THRESHOLD_MS:
        print("⚠️  WARNING: Prediction times are high - consider model optimization")

if track_response_times:
    print("\n--- BIRD TRACKING RESPONSE TIMES ---")
    for track_id, times in track_response_times.items():
        pattern = track_patterns.get(track_id, "Unknown")
        print(f"Bird {track_id} ({pattern}):")
        print(f"  First detected at frame: {times['detection_frame']}")
        print(f"  First prediction at frame: {times['prediction_frame']}")
        print(f"  Response time: {times['response_time_frames']} frames")
        
        # Calculate response time in milliseconds (assuming constant FPS)
        if fps > 0:
            response_time_ms = (times['response_time_frames'] / fps) * 1000
            print(f"  Response time: {response_time_ms:.2f} ms")
        print()

print("\n=== PROCESSING COMPLETE ===")

--- Phase 1: Pre-Analysis of Bird Detections ---
Identified 83 unique bird tracks.

=== BIRD MOVEMENT ANALYSIS ===
Bird 1: Unknown
  Avg Velocity: 12.93 px/frame
  Velocity STD: 8.70 px/frame
  Avg Acceleration: 2.64 px/frame²
  Avg Jerk: 2.18 px/frame³
  Avg Turning Angle: 0.10 rad
  Frames Tracked: 52

Bird 2: Unknown
  Avg Velocity: 10.37 px/frame
  Velocity STD: 7.54 px/frame
  Avg Acceleration: 1.83 px/frame²
  Avg Jerk: 1.39 px/frame³
  Avg Turning Angle: 0.19 rad
  Frames Tracked: 105

Bird 3: Unknown
  Avg Velocity: 8.64 px/frame
  Velocity STD: 4.30 px/frame
  Avg Acceleration: 2.17 px/frame²
  Avg Jerk: 0.96 px/frame³
  Avg Turning Angle: 0.12 rad
  Frames Tracked: 10

Bird 4: Unknown
  Avg Velocity: 11.64 px/frame
  Velocity STD: 4.91 px/frame
  Avg Acceleration: 2.04 px/frame²
  Avg Jerk: 1.68 px/frame³
  Avg Turning Angle: 0.35 rad
  Frames Tracked: 62

Bird 5: Maneuvering
  Avg Velocity: 4.35 px/frame
  Velocity STD: 2.88 px/frame
  Avg Acceleration: 1.49 px/frame²
  Avg 

### 7-FPS But correct working


In [4]:
import os
import cv2
import numpy as np
import tensorflow as tf
from ultralytics import YOLO
from collections import defaultdict
import random
import time
import argparse

# =============================================================================
# --- Configuration & Parameters ---
# =============================================================================

# Model & Sequence Parameters
SEQUENCE_LENGTH = 15
PREDICTION_HORIZON = 5

# Performance Optimization
# Run prediction every N frames for a given track to reduce computational load.
# A value of 1 predicts on every frame where history is sufficient.
# A value of 2 predicts on every other frame, etc.
PRED_EVERY_N_FRAMES = 1

# Visualization Parameters
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.5
BOX_THICKNESS = 1
PREDICTION_THICKNESS = 2
DOT_RADIUS = 3
TEXT_THICKNESS = 1
COLOR_PALETTE_SIZE = 50


# =============================================================================
# --- Helper Class: TrackBuffer ---
# =============================================================================

class TrackBuffer:
    """A fixed-size ring buffer to efficiently store position history for each track."""
    def __init__(self, seq_len):
        self.seq_len = seq_len
        self.buf = np.zeros((seq_len, 2), dtype=np.float32)
        self.ptr = 0
        self.count = 0

    def append(self, pos):
        """Adds a new position to the buffer."""
        self.buf[self.ptr] = pos
        self.ptr = (self.ptr + 1) % self.seq_len
        if self.count < self.seq_len:
            self.count += 1

    def is_full(self):
        """Checks if the buffer has enough history for a prediction."""
        return self.count == self.seq_len

    def get_ordered(self):
        """Returns the stored positions in chronological order."""
        if self.count < self.seq_len:
            return self.buf[:self.count].copy()
        # If full, the pointer is at the oldest entry, so we roll the array
        return np.roll(self.buf, -self.ptr, axis=0)


# =============================================================================
# --- Main Class: RealTimeBirdPredictor ---
# =============================================================================

class RealTimeBirdPredictor:
    """
    Integrates YOLO tracking and TFLite prediction for real-time video processing.
    """
    def __init__(self, yolo_path, tflite_path, conf_threshold=0.3):
        print("Initializing the real-time bird prediction system... 🚀")
        
        # 1. Load Object Detection & Tracking Model
        self.yolo_model = YOLO(yolo_path)
        print("✅ YOLOv8 model loaded.")

        # 2. Load Path Prediction Model (TFLite)
        try:
            self.interpreter = tf.lite.Interpreter(model_path=tflite_path)
            self.interpreter.allocate_tensors()
            self.input_details = self.interpreter.get_input_details()
            self.output_details = self.interpreter.get_output_details()
            self.input_index = self.input_details[0]['index']
            self.output_index = self.output_details[0]['index']
            print("✅ TFLite LSTM model loaded.")
        except Exception as e:
            raise ValueError(f"Error loading TFLite model: {e}")

        # 3. State Management & Configuration
        self.conf_threshold = conf_threshold
        self.track_buffers = defaultdict(lambda: TrackBuffer(SEQUENCE_LENGTH))
        self.track_predictions = {}
        self.track_colors = {}
        self.track_first_seen = {}
        self.color_palette = self._generate_colors(COLOR_PALETTE_SIZE)
        
        # For performance metrics display
        self.frame_times = []
        self.prediction_times = []

    def _generate_colors(self, num_colors):
        """Generates a list of random, visually distinct BGR colors."""
        rng = random.Random(42) # Seed for reproducibility
        colors = []
        for _ in range(num_colors):
            colors.append(tuple(rng.randint(60, 255) for _ in range(3)))
        return colors

    def _draw_prediction(self, frame, current_pos, predicted_points, color):
        """Draws the predicted future path on the frame."""
        if predicted_points is None or len(predicted_points) == 0:
            return
            
        # Create a list of all points: current + future
        all_points = [tuple(map(int, current_pos))] + [tuple(map(int, p)) for p in predicted_points]
        
        # Draw connecting lines
        for i in range(len(all_points) - 1):
            cv2.line(frame, all_points[i], all_points[i+1], color, PREDICTION_THICKNESS)
        
        # Draw a dot at the current position
        cv2.circle(frame, all_points[0], DOT_RADIUS + 1, color, -1)


    def process_stream(self, source, output_path=None):
        """
        Processes a video stream from a file or webcam for real-time prediction.
        
        Args:
            source (str or int): Path to video file or webcam ID (e.g., 0).
            output_path (str, optional): Path to save the output video.
        """
        # --- Video I/O Setup ---
        is_webcam = str(source).lower() == 'webcam'
        cap_source = 0 if is_webcam else source
        
        cap = cv2.VideoCapture(cap_source)
        if not cap.isOpened():
            raise IOError(f"Cannot open video source: {source}")
            
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        if is_webcam:
                TARGET_FPS = 30  # webcams are usually 30 (or set to 60 if supported)
        else:
            TARGET_FPS = cap.get(cv2.CAP_PROP_FPS)
            if TARGET_FPS <= 0 or TARGET_FPS > 120:  # fallback if metadata is wrong
                TARGET_FPS = 60
        fps = TARGET_FPS  # use for VideoWriter too

        fps = cap.get(cv2.CAP_PROP_FPS) if not is_webcam else 30

        out = None
        if output_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
            print(f"🎥 Output will be saved to: {output_path}")

        frame_number = 0
        
        # --- Main Processing Loop ---
        while True:
            t_start = time.perf_counter()
            ret, frame = cap.read()
            if not ret:
                print("End of video stream.")
                break

            # --- 1. Object Tracking ---
            results = self.yolo_model.track(
                frame, 
                persist=True, 
                conf=self.conf_threshold, 
                verbose=False
            )

            # --- 2. Process Detections & Predict ---
            active_track_ids = set()
            if results[0].boxes is not None and results[0].boxes.id is not None:
                track_ids = results[0].boxes.id.int().cpu().tolist()
                boxes = results[0].boxes.xyxy.cpu()
                active_track_ids.update(track_ids)

                for i, track_id in enumerate(track_ids):
                    box = boxes[i]
                    center_x = (box[0] + box[2]) / 2.0
                    center_y = (box[1] + box[3]) / 2.0
                    current_position = np.array([center_x, center_y], dtype=np.float32)

                    # Update track history
                    if track_id not in self.track_first_seen:
                        self.track_first_seen[track_id] = frame_number
                    self.track_buffers[track_id].append(current_position)
                    
                    # --- 3. Run Prediction ---
                    buffer = self.track_buffers[track_id]
                    if buffer.is_full() and (frame_number - self.track_first_seen[track_id]) % PRED_EVERY_N_FRAMES == 0:
                        history = buffer.get_ordered()
                        
                        # Normalize history
                        normalized_history = history / np.array([frame_width, frame_height], dtype=np.float32)
                        input_tensor = np.expand_dims(normalized_history, axis=0).astype(np.float32)
                        
                        # Run inference
                        t_pred_start = time.perf_counter()
                        self.interpreter.set_tensor(self.input_index, input_tensor)
                        self.interpreter.invoke()
                        pred_normalized = self.interpreter.get_tensor(self.output_index)
                        self.prediction_times.append((time.perf_counter() - t_pred_start) * 1000)
                        
                        # De-normalize and store prediction
                        pred_path = pred_normalized[0] * np.array([frame_width, frame_height], dtype=np.float32)
                        self.track_predictions[track_id] = pred_path
            
            # --- 4. Visualization ---
            for track_id in active_track_ids:
                color = self.track_colors.setdefault(track_id, self.color_palette[len(self.track_colors) % COLOR_PALETTE_SIZE])
                
                # Draw prediction if available
                if track_id in self.track_predictions:
                    history = self.track_buffers[track_id].get_ordered()
                    if history.shape[0] > 0:
                        self._draw_prediction(frame, history[-1], self.track_predictions[track_id], color)
                
                # Lightweight label
                history = self.track_buffers[track_id].get_ordered()
                if history.shape[0] > 0:
                    last_pos = history[-1]
                    label = f"ID:{track_id}"
                    cv2.putText(frame, label, (int(last_pos[0]) + 7, int(last_pos[1]) - 7), FONT, FONT_SCALE, color, TEXT_THICKNESS)

            # --- 5. Display Performance Metrics ---
            t_end = time.perf_counter()
            frame_time_ms = (t_end - t_start) * 1000
            self.frame_times.append(frame_time_ms)
            
            avg_frame_time = np.mean(self.frame_times[-30:]) if self.frame_times else 0
            avg_pred_time = np.mean(self.prediction_times[-30:]) if self.prediction_times else 0
            
            cv2.putText(frame, f"Frame Time: {avg_frame_time:.1f} ms (FPS: {1000/avg_frame_time:.1f})", (10, 20), FONT, FONT_SCALE, (0, 255, 0), TEXT_THICKNESS)
            cv2.putText(frame, f"Prediction Time: {avg_pred_time:.1f} ms", (10, 40), FONT, FONT_SCALE, (0, 255, 0), TEXT_THICKNESS)
            cv2.putText(frame, f"Active Birds: {len(active_track_ids)}", (10, 60), FONT, FONT_SCALE, (0, 255, 0), TEXT_THICKNESS)

            # --- 6. Show and Save Frame ---
            cv2.imshow("Real-Time Bird Prediction", frame)
            if out:
                out.write(frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("Interrupted by user.")
                break
            
            frame_number += 1
            
        # --- Cleanup ---
        cap.release()
        if out:
            out.release()
        cv2.destroyAllWindows()
        print("\n--- Processing Complete ---")
        
        # --- Final Performance Report ---
        if self.frame_times:
            avg_total_fps = 1000 / np.mean(self.frame_times)
            print(f"Average processing FPS: {avg_total_fps:.2f}")
            print(f"Average frame processing time: {np.mean(self.frame_times):.2f} ms")
        if self.prediction_times:
            print(f"Average TFLite prediction time: {np.mean(self.prediction_times):.2f} ms")


# =============================================================================
# --- Script Entry Point ---
# =============================================================================
# =============================================================================
# --- Script Entry Point (Modified for Notebook) ---
# =============================================================================

if __name__ == "__main__":
    # --- Hardcode your paths and settings here ---
    # Choose your source: a video file path or the string 'webcam'
    SOURCE_PATH = "D:/Project_P1/data/valid/test1.mp4" 
    # SOURCE_PATH = "webcam" 

    YOLO_MODEL_PATH = "D:/Project_P1/src/ml/runs/detect/train2/weights/best.pt"
    TFLITE_MODEL_PATH = "D:/Project_P1/results/video_analysis/model.tflite"
    OUTPUT_FILE_PATH = "D:/Project_P1/results/real_time_output_notebook.mp4" # Use None to disable saving
    CONFIDENCE_THRESHOLD = 0.3
    # ---------------------------------------------

    # --- Initialize and Run the Predictor ---
    try:
        predictor = RealTimeBirdPredictor(
            yolo_path=YOLO_MODEL_PATH,
            tflite_path=TFLITE_MODEL_PATH,
            conf_threshold=CONFIDENCE_THRESHOLD
        )
        predictor.process_stream(
            source=SOURCE_PATH,
            output_path=OUTPUT_FILE_PATH
        )
    except Exception as e:
        print(f"An error occurred: {e}")

Initializing the real-time bird prediction system... 🚀
✅ YOLOv8 model loaded.
✅ TFLite LSTM model loaded.
🎥 Output will be saved to: D:/Project_P1/results/real_time_output_notebook.mp4
Interrupted by user.

--- Processing Complete ---
Average processing FPS: 7.44
Average frame processing time: 134.34 ms
Average TFLite prediction time: 0.68 ms
